# Natural Language Processing (NLP) Course - Explained (موضح باللغة العربية)

هذا الملف يجمع كافة الدروس العملية والتمارين اللغوية للـ NLP موضحاً بالتفصيل باللغة العربية خطوة بخطوة.


## الدرس الأول: تنظيف النصوص (Text Cleaning)


### أولاً: استيراد المكتبات البرمجية المطلوبة (Import Libraries)

نقوم باستيراد مكتبة `re` للتعابير النمطية ومكتبة `html` لفك شفرات رموز الويب.


In [ ]:
# Step 1) استيراد المكتبات / Import libraries
import re
import html


### ثانياً: تعريف دالة تنظيف النصوص (Define clean_text function)

نقوم ببناء دالة متسلسلة تنظف النص خطوة بخطوة:
1. فك رموز HTML.
2. إزالة وسوم HTML.
3. إزالة الروابط والإيميلات وأرقام الهواتف.
4. إزالة المنشن والهاشتاق والرموز التعبيرية (Emoji).
5. إزالة الحروف غير الأبجدية وتوحيد الفراغات.


In [ ]:
# Step 2) تعريف دالة تنظيف النصوص / Define clean_text function
def clean_text(text):
    # 1. Decode HTML entities
    text = html.unescape(text)
    # 2. Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    # 3. Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # 4. Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    # 5. Remove phone numbers
    text = re.sub(r'\+?[\d\-\(\)\s]{9,}', '', text)
    # 6. Remove @mentions and #hashtags
    text = re.sub(r'@\w+|#\w+', '', text)
    # 7. Drop emoji/non-ASCII characters
    text = text.encode('ascii', 'ignore').decode('ascii')
    # 8. Keep only words, spaces, and basic sentence punctuation
    text = re.sub(r'[^\w\s\.\!\?]', ' ', text)
    # 9. Normalize extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


### ثالثاً: تجربة دالة التنظيف (Test cleaning function)

نقوم بتطبيق الدالة المعرفة على نص خام يحتوي على وسوم وروابط وإيموجي للتحقق من سلامة المخرجات قبل إدخالها للنموذج.


In [ ]:
# Step 3) تجربة دالة التنظيف / Test clean_text function
raw_sample = """
    <p>John said: "AI is amazing!!!! 🤖🔥"</p>
    Posted: 2026-07-01 | Source: @tech_blog
    Read more: https://example.com/article?id=123&ref=twitter
    Contact: +1-800-000-0000 | #ArtificialIntelligence #ML
"""
print("Before cleaning:")
print(raw_sample)
print("\nAfter cleaning:")
print(clean_text(raw_sample))


## الدرس الثاني: التوكننة والمعايرة (Tokenization & Normalization)


### أولاً: استيراد المكتبات وتنزيل الموارد (Import and Download)

نقوم باستيراد أدوات مكتبة NLTK ومكتبة SpaCy اللغوية وتنزيل حزم اللغات المعتمدة.


In [ ]:
# Step 1) استيراد المكتبات وتنزيل الملفات / Import libraries and download assets
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize, TweetTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import spacy

nltk.download('punkt')
nltk.download('stopwords')
nlp = spacy.load('en_core_web_sm')


### ثانياً: تقطيع النصوص لمستويات مختلفة (Tokenization strategies)

نقوم بتجربة التقطيع على مستوى الكلمات والجمل، ونقارن التقطيع العادي مع تقطيع التغريدات الذي يحافظ على الهاشتاقات والإيميلات.


In [ ]:
# Step 2) تجربة أنواع التقطيع / Test tokenization strategies
sample_text = "I'm running to the store! The weather is great, #fun. Contact: info@ai.com."

print("Word Tokenization:")
print(word_tokenize(sample_text))

print("\nSentence Tokenization:")
print(sent_tokenize(sample_text))

print("\nTweet Tokenization (preserves hashtags & emails):")
tweet_tok = TweetTokenizer()
print(tweet_tok.tokenize(sample_text))


### ثالثاً: إزالة كلمات التوقف (Stop Words removal)

نقوم بفلترة الكلمات الشائعة جداً (مثل أدوات التعريف والتوصيل: the, is, and) والتي تستهلك مساحة الذاكرة وتكلفة الاستدعاء دون إضافة سياق هام.


In [ ]:
# Step 3) إزالة كلمات التوقف / Stop Words removal
stop_words = set(stopwords.words('english'))
tokens = word_tokenize(sample_text.lower())
filtered_tokens = [w for w in tokens if w not in stop_words and w.isalnum()]
print("Tokens after stop words removal:")
print(filtered_tokens)


### رابعاً: مقارنة خوارزميات الـ Stemming والـ Lemmatization

نقوم باختبار أداء خوارزمية Porter لقص اللواحق (التقريبية) مقابل SpaCy لاستخراج الكلمات الأساسية الحقيقية (الدقيقة نحوياً).


In [ ]:
# Step 4) مقارنة الجذوع والكلمات الأساسية / Compare Stemming vs Lemmatization
stemmer = PorterStemmer()
words = ["running", "mice", "was", "better"]

print(f"{'Word':<12} | {'Stemming (Porter)':<20} | {'Lemmatization (SpaCy)':<20}")
print("-"*60)
for w in words:
    stem = stemmer.stem(w)
    lemma = nlp(w)[0].lemma_
    print(f"{w:<12} | {stem:<20} | {lemma:<20}")


## الدرس الثالث: تحليل النصوص والكيانات المسماة (Text Analysis & NER)


### أولاً: استيراد المكتبات وتهيئة النموذج اللغوي (Import and Init)

نقوم باستيراد مكتبة SpaCy وتحميل النموذج اللغوي الصغير للغة الإنجليزية.


In [ ]:
# Step 1) استيراد المكتبات والتهيئة / Import libraries
import spacy
nlp = spacy.load('en_core_web_sm')


### ثانياً: استخراج أجزاء الكلام وتحليل العلاقات النحوية (POS and Dependency Parsing)

نقوم بتحليل دور كل كلمة نحوياً (فاعل، اسم، فعل...) وعلاقتها بالكلمات الأخرى في الجملة لبناء فهم أعمق لبنية النص.


In [ ]:
# Step 2) استخراج أجزاء الكلام والعلاقات / Part-of-Speech and Dependency Parsing
sample_text = "Apple is looking at buying a U.K. startup for $1 billion."
doc = nlp(sample_text)

print(f"{'Text':<12} | {'POS Tag':<10} | {'Dependency':<12} | {'Explanation':<25}")
print("-"*65)
for token in doc:
    print(f"{token.text:<12} | {token.pos_:<10} | {token.dep_:<12} | {spacy.explain(token.pos_):<25}")


### ثالثاً: استخراج الكيانات المسماة (Named Entity Recognition)

نقوم برصد الأسماء الخاصة بالشركات، الدول، العملات، الأرقام والتواريخ وتصنيفها بشكل آلي.


In [ ]:
# Step 3) استخراج الكيانات المسمى / Named Entity Recognition (NER)
print("Named Entities found:")
for ent in doc.ents:
    print(f"Entity: {ent.text:<15} | Label: {ent.label_:<10} | Meaning: {spacy.explain(ent.label_)}")


### رابعاً: تطبيق عملي: استخراج جوانب مراجعات المنتجات (Aspect-Based Analyzer)

نقوم ببناء خوارزمية ذكية تحلل تعليقات العملاء، وتستخرج منها الميزة (Noun) والوصف المصاحب لها (Adjective) لمعرفة آراء المتسوقين في أجزاء المنتج بالتحديد.


In [ ]:
# Step 4) مشروع مصغر: استخراج مراجعات المنتجات / Aspect-Based Review Analyzer
def analyze_aspects(text):
    doc = nlp(text)
    aspects = {}
    for token in doc:
        if token.pos_ == "NOUN":
            adjectives = [child.text for child in token.children if child.pos_ == "ADJ"]
            if adjectives:
                aspects[token.text] = adjectives
    return aspects

reviews = [
    "The battery life is amazing, but the screen is too small.",
    "Great camera quality, but the phone gets very hot.",
    "Fast performance and beautiful display."
]

print("Aspect-based Review Analysis:")
for r in reviews:
    print(f"Review: '{r}'")
    print(f"Aspects: {analyze_aspects(r)}\n")


## الدرس الرابع: نمذجة اللغة الإحصائية (Language Modeling - N-grams)


### أولاً: استيراد المكتبات وتحضير النصوص التدريبية (Import and Prepare Data)

نقوم بتحضير مجموعة نصوص صغيرة وتقسيمها لكلمات لبناء النموذج عليها.


In [ ]:
# Step 1) استيراد المكتبات وتجهيز البيانات / Import libraries and define sample corpus
from collections import Counter, defaultdict
import numpy as np

corpus = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the cat ate the fish"
]
tokenized_corpus = [sentence.split() for sentence in corpus]
print("Tokenized Corpus:")
print(tokenized_corpus)


### ثانياً: حساب تكرار الكلمات الأحادية والثنائية (Build Unigram & Bigram Counts)

نقوم بحساب عدد مرات ظهور كل كلمة منفصلة (Unigrams) وعدد مرات ظهور كل كلمتين متتاليتين معاً (Bigrams) في كامل النصوص مع إضافة علامات بداية ونهاية الجمل.


In [ ]:
# Step 2) حساب التكرارات للأحاديات والثنائيات / Build unigram and bigram counts
unigrams = Counter()
bigrams = defaultdict(Counter)

for sentence in tokenized_corpus:
    # Add start and end tokens
    tokens = ["<s>"] + sentence + ["</s>"]
    for i in range(len(tokens) - 1):
        unigrams[tokens[i]] += 1
        bigrams[tokens[i]][tokens[i+1]] += 1
    unigrams[tokens[-1]] += 1

print("Unigram counts:")
print(dict(unigrams.most_common(5)))

print("\nBigram count examples (P(word | 'the')):")
print(dict(bigrams["the"]))


### ثالثاً: حساب احتمالية تتابع الكلمات (Laplace Bigram Probability)

نقوم بتطبيق معادلة احتمالية الأقصى (MLE) مع تنعيم لابلَس لمعالجة الكلمات النادرة التي لم تظهر في التدريب وحساب قيم الاحتمالات لها بنجاح.


In [ ]:
# Step 3) حساب احتمالية الكلمات مع Smoothing / Calculate probabilities with Laplace smoothing
def get_bigram_prob(w1, w2, alpha=1.0):
    count_w1 = unigrams[w1]
    count_w1_w2 = bigrams[w1].get(w2, 0)
    vocab_size = len(unigrams)
    # Laplace (Add-one) Smoothing formula
    prob = (count_w1_w2 + alpha) / (count_w1 + alpha * vocab_size)
    return prob

print("P(sat | cat) =", get_bigram_prob("cat", "sat"))
print("P(mat | the) =", get_bigram_prob("the", "mat"))
print("P(fish | dog) =", get_bigram_prob("dog", "fish"))
